In [1]:
import pandas as pd
import geopandas as gpd

In [2]:
from shapely.geometry import Polygon, MultiPolygon, GeometryCollection
from shapely.ops import unary_union

def extract_polygon_geom(geom):
    
    """Extract Polygon/MultiPolygon geometries from a GeometryCollection"""

    if geom is None:
        return None
    
    elif geom.geom_type in ['Polygon', 'MultiPolygon']:
        return geom
    
    elif geom.geom_type == 'GeometryCollection':
        polygons = [g for g in geom.geoms if g.geom_type in ['Polygon', 'MultiPolygon']]

        if not polygons:
            return None
        
        return unary_union(polygons)  # Merge into a MultiPolygon or Polygon
    
    else:
        return None  # Ignore Point, LineString, etc.

In [3]:
def population_weighted_variables_mapping(
    shp_source_path, shp_target_path,
    data_csv, population_csv,
    source_id_col,
    target_id_col,
    variable_fields,
    population_field
):
    """
    Project variables from a source LSOA geometry (2001 or 2011) to 2021 LSOA geometry
    using population-weighted interpolation based entirely on spatial overlay.
    
    Parameters:
    - shp_source_path: path to source LSOA shapefile
    - shp_target_path: path to target LSOA shapefile
    - data_csv: CSV with variables to project
    - population_csv: CSV with population per source LSOA
    - source_id_col: column name for source LSOA ID
    - target_id_col: column name for target LSOA ID
    - variable_fields: list of variable fields to interpolate
    - population_field: name of population field in population_csv

    Returns:
    - DataFrame with [target_id_col] and interpolated variables
    """
    # Load shapefiles and data
    gdf_src = gpd.read_file(shp_source_path)
    gdf_tgt = gpd.read_file(shp_target_path)
    data_df = pd.read_csv(data_csv)
    pop_df = pd.read_csv(population_csv)

    # Clean column names
    data_df.columns = data_df.columns.str.replace('\xa0', ' ', regex=True).str.strip()
    pop_df.columns = pop_df.columns.str.replace('\xa0', ' ', regex=True).str.strip()

    # Keep necessary columns
    data_df = data_df[[source_id_col] + variable_fields]
    pop_df = pop_df[[source_id_col, population_field]]

    # Merge attributes into source GeoDataFrame
    gdf_src = gdf_src.merge(data_df, on=source_id_col, how='left')
    gdf_src = gdf_src.merge(pop_df, on=source_id_col, how='left')

    # Check required columns
    required_cols = [source_id_col, population_field] + variable_fields
    
    for col in required_cols:
        if col not in gdf_src.columns:
            raise KeyError(f"Missing column '{col}' in source GeoDataFrame.")

    # Ensure matching CRS
    gdf_tgt = gdf_tgt.to_crs(gdf_src.crs)

    # Compute source area
    gdf_src_clone = gdf_src.copy()
    gdf_src['area_total'] = gdf_src_clone.geometry.area

    # Spatial overlay — keep all geometries to avoid loss
    intersections = gpd.overlay(gdf_src, gdf_tgt, how='intersection', keep_geom_type=False)

    intersections['geometry'] = intersections.geometry.apply(extract_polygon_geom)
    intersections = intersections[intersections.geometry.notnull()]

    intersections['area_ij'] = intersections.geometry.area

    intersections['pop_ij'] = (intersections['area_ij'] / intersections['area_total']) * intersections[population_field]

    for field in variable_fields:
        intersections[field] = intersections[field].fillna(0)
        intersections[f'{field}_weighted'] = intersections['pop_ij'] / intersections[population_field] * intersections[field]

    agg_fields = {f'{field}_weighted': 'sum' for field in variable_fields}
    agg_fields['pop_ij'] = 'sum'

    result = intersections.groupby(target_id_col).agg(agg_fields).reset_index()

    result.drop(columns=['pop_ij'], inplace=True)

    return result

In [26]:
df_num_of_sales = population_weighted_variables_mapping(
    shp_source_path = "../Output/London_LSOA_2011.shp",
    shp_target_path = "../Output/London_LSOA_2021.shp",
    data_csv = "../Output/Number of sales.csv",
    population_csv = "../Output/Population_2011.csv",
    source_id_col = "LSOA11CD",
    target_id_col = "LSOA21CD",
    variable_fields = ["Number of sales 2011", "Number of sales 2021"],
    population_field = "All residents"
)

In [27]:
df_num_of_sales = df_num_of_sales.round(0)

In [28]:
df_num_of_sales

,LSOA21CD,Number of sales 2011_weighted,Number of sales 2021_weighted
0,E01000001,141.0,123.0
1,E01000002,233.0,144.0
2,E01000003,168.0,95.0
3,E01000005,41.0,10.0
4,E01000006,44.0,58.0
...,...,...,...
4989,E01035718,189.0,113.0
4990,E01035719,75.0,41.0
4991,E01035720,84.0,46.0
4992,E01035721,100.0,89.0


In [29]:
df_num_of_sales.to_csv("../Output/Number of sales mapping.csv", index=False)

In [30]:
df_imd_2019_ = population_weighted_variables_mapping(
    shp_source_path = "../Output/London_LSOA_2011.shp",
    shp_target_path = "../Output/London_LSOA_2021.shp",
    data_csv = "../Output/IMD_london_2019.csv",
    population_csv = "../Output/Population_2011.csv",
    source_id_col = "LSOA11CD",
    target_id_col = "LSOA21CD",
    variable_fields = ["Index of Multiple Deprivation (IMD) Score"],
    population_field = "All residents"
)

In [31]:
df_imd_2019_

,LSOA21CD,Index of Multiple Deprivation (IMD) Score_weighted
0,E01000001,6.208000
1,E01000002,5.143049
2,E01000003,19.401951
3,E01000005,28.652000
4,E01000006,19.837000
...,...,...
4989,E01035718,21.282000
4990,E01035719,7.593251
4991,E01035720,8.581749
4992,E01035721,56.179001


In [32]:
df_imd_2019_.to_csv("../Output/IMD2019 mapping.csv", index=False)

In [33]:
df_imd_2010_ = population_weighted_variables_mapping(
    shp_source_path = "../Output/London_LSOA_2001.shp",
    shp_target_path = "../Output/London_LSOA_2021.shp",
    data_csv = "../Output/IMD_london_2010.csv",
    population_csv = "../Output/Population_2001.csv",
    source_id_col = "LSOA01CD",
    target_id_col = "LSOA21CD",
    variable_fields = ["IMD SCORE"],
    population_field = "All people"
)

In [34]:
df_imd_2010_

,LSOA21CD,IMD SCORE_weighted
0,E01000001,6.161643
1,E01000002,5.585180
2,E01000003,13.292872
3,E01000005,21.364986
4,E01000006,17.077753
...,...,...
4989,E01035718,24.200239
4990,E01035719,8.494673
4991,E01035720,9.599883
4992,E01035721,49.680054


In [35]:
df_imd_2010_.to_csv("../Output/IMD2010 mapping.csv", index=False)

In [36]:
tenure_2011 = population_weighted_variables_mapping(
    shp_source_path = "../Output/London_LSOA_2011.shp",
    shp_target_path = "../Output/London_LSOA_2021.shp",
    data_csv = "../Dataset/Census London 2011/Tenure_2011.csv",
    population_csv = "../Output/Population_2011.csv",
    source_id_col = "LSOA11CD",
    target_id_col = "LSOA21CD",
    variable_fields = ['All households', 'Owned',
       'Owned: Owned outright', 'Owned: Owned with a mortgage or loan',
       'Shared ownership (part owned and part rented)', 'Social rented',
       'Social rented: Rented from council (Local Authority)',
       'Social rented: Other', 'Private rented',
       'Private rented: Private landlord or letting agency',
       'Private rented: Other', 'Living rent free'],
    population_field = "All residents"
)

tenure_2011 = tenure_2011.round(0)

In [37]:
tenure_2011.to_csv("../Dataset/Census London 2011/Tenure_2011 mapping.csv", index=False)

In [38]:
occupation_2011 = population_weighted_variables_mapping(
    shp_source_path = "../Output/London_LSOA_2011.shp",
    shp_target_path = "../Output/London_LSOA_2021.shp",
    data_csv = "../Dataset/Census London 2011/Occupation_2011.csv",
    population_csv = "../Output/Population_2011.csv",
    source_id_col = "LSOA11CD",
    target_id_col = "LSOA21CD",
    variable_fields = ['All categories: Occupation',
       '1. Managers, directors and senior officials',
       '2. Professional occupations',
       '3. Associate professional and technical occupations',
       '4. Administrative and secretarial occupations',
       '5. Skilled trades occupations',
       '6. Caring, leisure and other service occupations',
       '7. Sales and customer service occupations',
       '8. Process plant and machine operatives', '9. Elementary occupations'],
    population_field = "All residents"
)

occupation_2011 = occupation_2011.round(0)

In [39]:
occupation_2011.to_csv("../Dataset/Census London 2011/Occupation_2011 mapping.csv", index=False)

In [40]:
ethnic_2011 = population_weighted_variables_mapping(
    shp_source_path = "../Output/London_LSOA_2011.shp",
    shp_target_path = "../Output/London_LSOA_2021.shp",
    data_csv = "../Dataset/Census London 2011/Ethnic_group_2011.csv",
    population_csv = "../Output/Population_2011.csv",
    source_id_col = "LSOA11CD",
    target_id_col = "LSOA21CD",
    variable_fields = ['All usual residents', 'White',
       'White: English/Welsh/Scottish/Northern Irish/British', 'White: Irish',
       'White: Gypsy or Irish Traveller', 'White: Other White',
       'Mixed/multiple ethnic groups',
       'Mixed/multiple ethnic groups: White and Black Caribbean',
       'Mixed/multiple ethnic groups: White and Black African',
       'Mixed/multiple ethnic groups: White and Asian',
       'Mixed/multiple ethnic groups: Other Mixed', 'Asian/Asian British',
       'Asian/Asian British: Indian', 'Asian/Asian British: Pakistani',
       'Asian/Asian British: Bangladeshi', 'Asian/Asian British: Chinese',
       'Asian/Asian British: Other Asian',
       'Black/African/Caribbean/Black British',
       'Black/African/Caribbean/Black British: African',
       'Black/African/Caribbean/Black British: Caribbean',
       'Black/African/Caribbean/Black British: Other Black',
       'Other ethnic group', 'Other ethnic group: Arab',
       'Other ethnic group: Any other ethnic group'],
    population_field = "All residents"
)

ethnic_2011 = ethnic_2011.round(0)

In [41]:
ethnic_2011.to_csv("../Dataset/Census London 2011/Ethnic_group_2011 mapping.csv", index=False)

In [42]:
age_2011 = population_weighted_variables_mapping(
    shp_source_path = "../Output/London_LSOA_2011.shp",
    shp_target_path = "../Output/London_LSOA_2021.shp",
    data_csv = "../Dataset/Census London 2011/Age_structure_2011.csv",
    population_csv = "../Output/Population_2011.csv",
    source_id_col = "LSOA11CD",
    target_id_col = "LSOA21CD",
    variable_fields = ['Age 0 to 4', 'Age 5 to 7', 
       'Age 8 to 9', 'Age 10 to 14', 'Age 15', 'Age 16 to 17',
       'Age 18 to 19', 'Age 20 to 24', 'Age 25 to 29', 'Age 30 to 44',
       'Age 45 to 59', 'Age 60 to 64', 'Age 65 to 74', 'Age 75 to 84',
       'Age 85 to 89', 'Age 90 and over'],
    population_field = "All residents"
)

age_2011 = age_2011.round(0)

In [43]:
age_2011.to_csv("../Dataset/Census London 2011/Age_structure_2011 mapping.csv", index=False)

In [4]:
Education_2011 = population_weighted_variables_mapping(
    shp_source_path = "../Output/London_LSOA_2011.shp",
    shp_target_path = "../Output/London_LSOA_2021.shp",
    data_csv = "../Dataset/Census London 2011/Education_2011.csv",
    population_csv = "../Output/Population_2011.csv",
    source_id_col = "LSOA11CD",
    target_id_col = "LSOA21CD",
    variable_fields = ['All categories: Highest level of qualification', 'Highest level of qualification: Level 4 qualifications and above'],
    population_field = "All residents"
)

Education_2011 = Education_2011.round(0)

In [5]:
Education_2011.to_csv("../Dataset/Census London 2011/Education_2011 mapping.csv", index=False)